In [ ]:
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 12.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl



def compute_wda(kkt, khull, kcomp, kturb):
    khull_score = 2.0 - khull  # invert khull (higher = worse for this one)
    wda = 0.25*kkt + 0.10*khull_score + 0.35*kcomp + 0.30*kturb
    return wda

wda_range    = np.arange(0.930, 1.001, 0.0001)
rul_range    = np.arange(0, 101, 1)
health_range = np.arange(0, 1.01, 0.01)



wda    = ctrl.Antecedent(wda_range, 'wda')
rul    = ctrl.Antecedent(rul_range, 'rul')
health = ctrl.Consequent(health_range, 'health')


wda['poor']    = fuzz.trapmf(wda.universe, [0.930, 0.930, 0.945, 0.955])
wda['average'] = fuzz.trimf(wda.universe, [0.948, 0.963, 0.978])
wda['good']    = fuzz.trapmf(wda.universe, [0.972, 0.984, 1.000, 1.000])

rul['poor']    = fuzz.trapmf(rul.universe, [0, 0, 20, 35])
rul['average'] = fuzz.trimf(rul.universe, [20, 50, 80])
rul['good']    = fuzz.trapmf(rul.universe, [65, 80, 100, 100])

health['poor']    = fuzz.trapmf(health.universe, [0.00, 0.00, 0.25, 0.40])
health['average'] = fuzz.trimf(health.universe, [0.30, 0.50, 0.70])
health['good']    = fuzz.trapmf(health.universe, [0.60, 0.75, 1.00, 1.00])


rules = [
    ctrl.Rule(wda['poor']    & rul['poor'],    health['poor']),
    ctrl.Rule(wda['poor']    & rul['average'], health['poor']),
    ctrl.Rule(wda['poor']    & rul['good'],    health['average']),
    ctrl.Rule(wda['average'] & rul['poor'],    health['poor']),
    ctrl.Rule(wda['average'] & rul['average'], health['average']),
    ctrl.Rule(wda['average'] & rul['good'],    health['average']),
    ctrl.Rule(wda['good']    & rul['poor'],    health['average']),
    ctrl.Rule(wda['good']    & rul['average'], health['good']),
    ctrl.Rule(wda['good']    & rul['good'],    health['good']),
]



system = ctrl.ControlSystem(rules)
sim    = ctrl.ControlSystemSimulation(system)

#testing 
kkt   = 0.92
khull = 1.15
kcomp = 0.96
kturb = 0.98
rul_v = 25.0

wda_val = compute_wda(kkt, khull, kcomp, kturb)


wda_val = np.clip(wda_val, 0.930, 1.000)
rul_v   = np.clip(rul_v, 0, 100)#

sim.input['wda'] = wda_val
sim.input['rul'] = rul_v
sim.compute()

score = sim.output['health']

# fuzzy-consistent label assignment
poor_deg = fuzz.interp_membership(health.universe, health['poor'].mf, score)
avg_deg  = fuzz.interp_membership(health.universe, health['average'].mf, score)
good_deg = fuzz.interp_membership(health.universe, health['good'].mf, score)

if poor_deg >= avg_deg and poor_deg >= good_deg:
    label = 'POOR'
elif avg_deg >= poor_deg and avg_deg >= good_deg:
    label = 'AVERAGE'
else:
    label = 'GOOD'

print(f"WDA   : {wda_val:.4f}")
print(f"Score : {score:.3f}")
print(f"Health: {label}")
print(f"Memberships → Poor:{poor_deg:.3f}, Average:{avg_deg:.3f}, Good:{good_deg:.3f}")

WDA   : 0.9450
Score : 0.176
Health: POOR
Memberships → Poor:1.000, Average:0.000, Good:0.000
